# Exploratory Data Analysis

In [1]:
# FNSPID Description
#TODO

In [2]:
import pandas as pd
import numpy as np
import os
import tiktoken

In [3]:
# FNSPID Fields

The dataset is 22GB. Way too big for my computer's memory! For that reason, we use a sample of 100,000 rows for the exploratory data analysis.

In [4]:
df_sample = pd.read_csv("FNSPID_data\\nasdaq_external_data.csv", nrows=100000)


In [5]:
df_sample.columns

Index(['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url',
       'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary',
       'Textrank_summary', 'Lexrank_summary'],
      dtype='object')

There are fairly few columns in the dataset. For that reason, we can afford to consider them one by one. But first, let's check whether any of the columns have missing values. 

In [6]:
# Count the number of missing by column
missing = df_sample.isnull().sum()
print(missing)

Unnamed: 0               0
Date                     0
Article_title            0
Stock_symbol             0
Url                      0
Publisher           100000
Author              100000
Article                  0
Lsa_summary              1
Luhn_summary             0
Textrank_summary         0
Lexrank_summary          0
dtype: int64


All 100,000 samples having Publisher and Author set to Null. We can ignore those two fields.

In [7]:
df_sample['Unnamed: 0']

0            0.0
1            1.0
2            2.0
3            3.0
4            4.0
          ...   
99995    99995.0
99996    99996.0
99997    99997.0
99998    99998.0
99999    99999.0
Name: Unnamed: 0, Length: 100000, dtype: float64

This first column is the index column, which is not needed. We can ignore it.

In [8]:
df_sample['Date'].head(5)

0    2023-12-16 23:00:00 UTC
1    2023-12-12 00:00:00 UTC
2    2023-12-12 00:00:00 UTC
3    2023-12-07 00:00:00 UTC
4    2023-12-07 00:00:00 UTC
Name: Date, dtype: object

This column indicates the date of publication of the article. This will be useful for the train/validation/split: we can do a temporal split, with the oldest articles in the train data, to avoid leakage. 

In [9]:
df_sample['Article_title'].head(5)

0    Interesting A Put And Call Options For August ...
1    Wolfe Research Initiates Coverage of Agilent T...
2    Agilent Technologies Reaches Analyst Target Price
3    Agilent (A) Enhances BioTek Cytation C10 With ...
4    Pre-Market Most Active for Dec 7, 2023 : SQQQ,...
Name: Article_title, dtype: object

This column contains the titles of the articles. Although this is text, we will focus on training the model to write the body of articles rather than their title, so we drop that column. 

In [10]:
df_sample['Stock_symbol'].unique()

array(['A', 'AA', 'AAAU', 'AACG', 'AADR', 'AAL', 'AAMC', 'AAME', 'AAN',
       'AAOI', 'AAON', 'AAP', 'AAPL', 'AAT', 'AAU', 'AAXJ', 'AB', 'ABBV',
       'ABCB', 'ABEO', 'ABEQ', 'ABEV', 'ABG', 'ABIO', 'ABM', 'ABR', 'ABT',
       'ABUS', 'AC', 'ACA', 'ACAD', 'ACB', 'ACCO', 'ACEL', 'ACES', 'ACGL',
       'ACGLO', 'ACHC', 'ACHV', 'ACI', 'ACIO', 'ACIU', 'ACIW', 'ACLS',
       'ACM', 'ACMR', 'ACN', 'ACNB', 'ACOR', 'ACP', 'ACRE', 'ACRS',
       'ACRX', 'ACSI', 'ACST', 'ACT', 'ACTG', 'ACU', 'ACV', 'ACWI',
       'ACWV', 'ACWX', 'ADAP', 'ADBE', 'ADC', 'ADCT', 'ADES', 'ADI',
       'ADIL', 'ADM', 'ADMA', 'ADME', 'ADNT', 'ADP', 'ADPT', 'ADSK',
       'ADT', 'ADTN', 'ADTX', 'ADUS', 'ADVM', 'ADX', 'ADXN', 'ADXS', 'AE',
       'AEE', 'AEF', 'AEFC', 'AEG', 'AEHR', 'AEIS', 'AEL', 'AEM', 'AEMD',
       'AEO', 'AEP', 'AER', 'AES', 'AESR', 'AEY', 'AEYE', 'AEZS', 'AFB',
       'AFG', 'AFGB', 'AFGC', 'AFGD', 'AFIF', 'AFK', 'AFL', 'AFLG',
       'AFMC', 'AFMD', 'AFSM', 'AFT', 'AFTY', 'AFYA', 'AG', 'AGBA',
 

That field contains the stock ticker associated with each article. We initially considered using this field for the train/test/validation split, but decided that a temporal split is sufficient. Hence, we drop the ticker field.

In [11]:
df_sample['Url'].head(5)

0    https://www.nasdaq.com/articles/interesting-a-...
1    https://www.nasdaq.com/articles/wolfe-research...
2    https://www.nasdaq.com/articles/agilent-techno...
3    https://www.nasdaq.com/articles/agilent-a-enha...
4    https://www.nasdaq.com/articles/pre-market-mos...
Name: Url, dtype: object

The url of articles are not useful to our model, so we drop it.

In [12]:
df_sample['Article'][0]

'Investors in Agilent Technologies, Inc. (Symbol: A) saw new options begin trading this week, for the August 2024 expiration. One of the key inputs that goes into the price an option buyer is willing to pay, is the time value, so with 241 days until expiration the newly trading contracts represent a possible opportunity for sellers of puts or calls to achieve a higher premium than would be available for the contracts with a closer expiration. At Stock Options Channel, our YieldBoost formula has looked up and down the A options chain for the new August 2024 contracts and identified one put and one call contract of particular interest.\nThe put contract at the $125.00 strike price has a current bid of $4.50. If an investor was to sell-to-open that put contract, they are committing to purchase the stock at $125.00, but will also collect the premium, putting the cost basis of the shares at $120.50 (before broker commissions). To an investor already interested in purchasing shares of A, tha

The Article is the key field of our dataset. It contains the text of the financial article. This will be the field we will be using to train our language model. The example above is the text coming from one article. The full dataset contains millions like these.


As we can see below, the remaining four fields are different type of summary of the main article. This would not necessarily add value to our training, and we already have a gigantic dataset, so we ignore those fields.

In [13]:
df_sample['Lsa_summary'][0]


"Because the $125.00 strike represents an approximate 10% discount to the current trading price of the stock (in other words it is out-of-the-money by that percentage), there is also the possibility that the put contract would expire worthless. Of course, a lot of upside could potentially be left on the table if A shares really soar, which is why looking at the trailing twelve month trading history for Agilent Technologies, Inc., as well as studying the business fundamentals becomes important. Below is a chart showing A's trailing twelve month trading history, with the $150.00 strike highlighted in red: Considering the fact that the $150.00 strike represents an approximate 8% premium to the current trading price of the stock (in other words it is out-of-the-money by that percentage), there is also the possibility that the covered call contract would expire worthless, in which case the investor would keep both their shares of stock and the premium collected."

In [14]:
df_sample['Luhn_summary'][0]


'The current analytical data (including greeks and implied greeks) suggest the current odds of that happening are 77%. Below is a chart showing the trailing twelve month trading history for Agilent Technologies, Inc., and highlighting in green where the $125.00 strike is located relative to that history: Turning to the calls side of the option chain, the call contract at the $150.00 strike price has a current bid of $8.10. The current analytical data (including greeks and implied greeks) suggest the current odds of that happening are 53%.'

In [15]:
df_sample['Textrank_summary'][0]

"Below is a chart showing the trailing twelve month trading history for Agilent Technologies, Inc., and highlighting in green where the $125.00 strike is located relative to that history: Turning to the calls side of the option chain, the call contract at the $150.00 strike price has a current bid of $8.10. Below is a chart showing A's trailing twelve month trading history, with the $150.00 strike highlighted in red: Considering the fact that the $150.00 strike represents an approximate 8% premium to the current trading price of the stock (in other words it is out-of-the-money by that percentage), there is also the possibility that the covered call contract would expire worthless, in which case the investor would keep both their shares of stock and the premium collected. On our website under the contract detail page for this contract, Stock Options Channel will track those odds over time to see how they change and publish a chart of those numbers (the trading history of the option cont

In [16]:
df_sample['Lexrank_summary'][0]

"At Stock Options Channel, our YieldBoost formula has looked up and down the A options chain for the new August 2024 contracts and identified one put and one call contract of particular interest. Should the contract expire worthless, the premium would represent a 3.60% return on the cash commitment, or 5.45% annualized — at Stock Options Channel we call this the YieldBoost. Below is a chart showing A's trailing twelve month trading history, with the $150.00 strike highlighted in red: Considering the fact that the $150.00 strike represents an approximate 8% premium to the current trading price of the stock (in other words it is out-of-the-money by that percentage), there is also the possibility that the covered call contract would expire worthless, in which case the investor would keep both their shares of stock and the premium collected."

In [26]:
df_sample['article_length'] = df_sample['Article'].str.len()

print("Length in characters of longest article: ", df_sample['article_length'].max())
print("Length in characters of shortest article: ", df_sample['article_length'].min())
print("Length in characters of median article: ", int(df_sample['article_length'].median()))

Length in characters of longest article:  272968
Length in characters of shortest article:  1
Length in characters of median article:  3955


# Data Transformations

We can now start preparing the train, test, and validation datasets.

Following the Exploratory Data Analysis above, we wil use only two columns: 'Data' for the train/validation/test split and 'Article' for the text data.

In [27]:
df = pd.read_csv("FNSPID_data\\nasdaq_external_data.csv", usecols=['Date','Article'])

C:\Users\hugol\AppData\Local\Temp\ipykernel_23816\3859709655.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("FNSPID_data\\nasdaq_external_data.csv", usecols=['Date','Article'])


In [28]:
# Force the correct types
df['Date'] = pd.to_datetime(df['Date'])
df['Article'] = df['Article'].astype(str)


## Cleanup the Dataset

We delete a row if any of the following two condition is true.
-  Any of the two columns of interest is missing.
- The length of Article is less than 100 characters. 

The character threshold is selected to be 100, because anything less than that would be too short to contain financial news information, and is probably something like an error message, a disclaimer, or some other type of brief message we want to ignore.


In [29]:

df['Missing'] = df.apply(lambda x: 1 if (pd.isnull(x['Date']) or pd.isnull(x['Article']) or len(x['Article']) < 100) else 0, axis=1)
missing_count = df['Missing'].value_counts()[1]
print(f"There are {missing_count} missing values in the dataset.")

There are 11811438 missing values in the dataset.


In [30]:
df = df.dropna(subset=['Date', 'Article'])
df = df.drop(columns=['Missing'])

In [31]:
print("After removing missings, there are ", len(df), "rows remaining in the dataset.")

After removing missings, there are  15549299 rows remaining in the dataset.


## Train-Validation-Test Split

In [ ]:
# Oldest published data
oldest_date = df['Date'].min()
print("Oldest published data: ", oldest_date)

# Newest published data
newest_date = df['Date'].max()
print("Newest published data: ", newest_date)

For timestamped data, it is always good to have held-out data be as of a later date than the training data. This way, there cannot be leakage of information from the held-out sets to the training set. 

We split the rows according to the following rule:
- Articles published in 2021 and before are in the training dataset.
- Articles published in 2022 are in the validation dataset.
- Articles published in 2023 are in the test dataset.  

In [32]:

df['Set'] = np.where(df['Date'] >= '2023-01-01', 'Test',
                     np.where(df['Date'] >= '2022-01-01', 'Validation', 'Train'))

train = df[df['Set'] == 'Train']
train = train.drop(columns=['Date','Set'])

validation = df[df['Set'] == 'Validation']
validation = validation.drop(columns=['Date','Set'])

test = df[df['Set'] == 'Test']
test = test.drop(columns=['Date','Set'])

print("The train dataset contains", len(train), "articles.")
print("The validation dataset contains", len(validation), "articles.")
print("The test dataset contains", len(test), "articles.")

The train dataset contains 14314758 articles.
The validation dataset contains 280354 articles.
The test dataset contains 954187 articles.


In [33]:
train.head(1)

,Article
923,"MADRID, Dec 30 (Reuters) - Alcoa AA.N has line..."


In [34]:
df.head(1)

,Date,Article,Set
0,2023-12-16 23:00:00+00:00,"Investors in Agilent Technologies, Inc. (Symbo...",Test


In [35]:
train.head()

,Article
923,"MADRID, Dec 30 (Reuters) - Alcoa AA.N has line..."
924,"Adds quote, detail, background\nOSLO, Dec 30 (..."
925,"Updates prices, adds comment\nDec 30 (Reuters)..."
926,"InvestorPlace - Stock Market News, Stock Advic..."
927,"MADRID, Dec 29 (Reuters) - Alcoa Corp AA.N rea..."


In [36]:

validation.head()

,Article
304,Agilent Technologies (A) is one of the stocks ...
305,Alphabet’s GOOGL division Google is consistent...
306,Airbnb ABNB has made an announcement to stop p...
307,Looking at the universe of stocks we cover at ...
308,Agilent Technologies (A) could be a solid choi...


In [37]:
test.head()


,Article
0,"Investors in Agilent Technologies, Inc. (Symbo..."
1,"Fintel reports that on December 13, 2023, Wolf..."
2,"In recent trading, shares of Agilent Technolog..."
3,Agilent Technologies A is enhancing its BioTek...
4,The NASDAQ 100 Pre-Market Indicator is up 70.2...


## Convert Text to Tokens


Our model needs a tokenizer to convert tokens into numbers and vice versa. We will use the tiktoken library to do this. We first convert each dataset into a long string of text. Next, we apply the tokenizer to convert those long strings into lists of integers (each integer representing a token). Since this is a very compute-intensive process for such a large dataset, we repeatedly save and load from file to avoid having to run again if it crashes.

In [38]:
def save_string_to_file(content, filename):
    with open( os.path.join("FNSPID_transformed", filename), 'w', encoding='utf-8') as file:
        file.write(content)

In [ ]:
def load_string_from_file(filename):
    with open(os.path.join("FNSPID_transformed", filename), 'r', encoding='utf-8') as file:
        content = file.read()
    return content

In [39]:
# Combine the entire train['Article'] into a single string, and save as .npy file
train_articles = ' '.join(train['Article'].tolist())
save_string_to_file(train_articles, 'train_articles.txt')


In [40]:
# Combine the entire validation['Article'] into a single string, and save as .npy file
validation_articles = ' '.join(validation['Article'].tolist())
save_string_to_file(validation_articles, 'validation_articles.txt')

In [41]:
# Combine the entire test['Article'] into a single string, and save as .npy file
test_articles = ' '.join(test['Article'].tolist())
save_string_to_file(test_articles, 'test_articles.txt')

In [ ]:
# Open-source tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
train_articles = load_string_from_file('train_articles.txt')
train_tokens = tokenizer.encode(train_articles)
np.save(os.path.join("FNSPID_transformed", 'train_tokens.npy'), train_tokens)

In [ ]:
validation_articles = load_string_from_file('validation_articles.txt')
validation_tokens = tokenizer.encode(validation_articles)
np.save(os.path.join("FNSPID_transformed", 'validation_tokens.npy'), validation_tokens)

In [ ]:
test_articles = load_string_from_file('test_articles.txt')
test_tokens = tokenizer.encode(test_articles)
np.save(os.path.join("FNSPID_transformed", 'test_tokens.npy'), test_tokens)

In [ ]:
print("Train tokens count:", len(train_tokens))
print("Validation tokens count:", len(validation_tokens))
print("Test tokens count:", len(test_tokens))

In [ ]:
# Save the tokenized data as .npy files
np.save(os.path.join("FNSPID_transformed", 'train_tokens_BACKUP.npy'), train_tokens)
np.save(os.path.join("FNSPID_transformed", 'validation_tokens_BACKUP.npy'), validation_tokens)
np.save(os.path.join("FNSPID_transformed", 'test_tokens_BACKUP.npy'), test_tokens)

In [ ]:
# Feature Engineering - Not required